# Tokenization in Generative AI

Tokenization is a fundamental step in how Large Language Models (LLMs) process text. Before a model can understand or generate language, raw text must be converted into a sequence of numeric **tokens**.

This notebook explores tokenization through three hands-on examples:

1. **GPT-2 Tokenizer** — Using a pretrained Hugging Face tokenizer to tokenize, encode, and decode text
2. **Inspecting BPE Artifacts** — Locating the vocabulary and merge rules that define a tokenizer
3. **Custom BPE Tokenizer** — Training a Byte Pair Encoding tokenizer from scratch on your own data

---

## 1. GPT-2 Tokenizer — Tokenize, Encode & Decode

GPT-2 uses **Byte Pair Encoding (BPE)**, a subword tokenization algorithm. BPE starts with a character-level vocabulary and iteratively merges the most frequent adjacent pairs until the target vocabulary size is reached.

Key operations demonstrated below:

| Method | What it does |
|--------|-------------|
| `tokenize()` | Splits text into subword token strings |
| `convert_tokens_to_ids()` | Maps token strings to integer IDs |
| `tokenizer(text)` | Encodes text and returns `input_ids` + `attention_mask` |
| `decode()` | Reconstructs text from a list of token IDs |

> **Note:** The `Ġ` prefix on tokens (e.g. `Ġhow`) indicates a leading space — GPT-2 encodes whitespace as part of the token to preserve word boundaries.

In [9]:
from transformers import GPT2Tokenizer

# Load the pretrained GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Example text to tokenize
text = "Hello, how are you doing today?"

# Tokenize the text
tokens = tokenizer.tokenize(text)

# Convert tokens to their corresponding IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)


# print the results
print("Original Text:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

# Optional: Encode directly to token IDs using the tokenizer
encoded = tokenizer(text)
print("Encoded Token IDs:", encoded)

# Decode the token IDs back to text
decoded_text = tokenizer.decode(token_ids)
print("Decoded Text:", decoded_text)

Original Text: Hello, how are you doing today?
Tokens: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', 'Ġdoing', 'Ġtoday', '?']
Token IDs: [15496, 11, 703, 389, 345, 1804, 1909, 30]
Encoded Token IDs: {'input_ids': [15496, 11, 703, 389, 345, 1804, 1909, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Decoded Text: Hello, how are you doing today?


## 2. Inspecting GPT-2 Vocabulary & Merge Rules

Every BPE tokenizer is defined by two artifact files that Hugging Face caches locally:

| File | Purpose |
|------|---------|
| `vocab.json` | Maps token strings → integer IDs (the full vocabulary) |
| `merges.txt` | Ordered list of BPE merge rules learned during training |

`GPT2TokenizerFast` is the Rust-backed fast variant — same results as `GPT2Tokenizer` but significantly faster on large texts. Here we use `cached_file()` to locate the underlying files in the Hugging Face cache so you can inspect or reuse them.

In [10]:
from transformers import GPT2TokenizerFast
from transformers.utils.hub import cached_file

# Load the pretrained GPT-2 tokenizer
tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

# Get the vocab, merges file path using Hugging Face's cached_file function
vocab_file_path = cached_file("gpt2", "vocab.json")
merges_file_path = cached_file("gpt2", "merges.txt")

print("Vocab file path:", vocab_file_path)
print("Merges file path:", merges_file_path)

Vocab file path: /home/m.nushath/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e/vocab.json
Merges file path: /home/m.nushath/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e/merges.txt


## 3. Training a Custom BPE Tokenizer from Scratch

Instead of loading a pretrained tokenizer, we can train one on our own corpus using Hugging Face's low-level `tokenizers` library (Rust-backed).

**Training pipeline:**

| Step | Component | Role |
|------|-----------|------|
| 1 | `BPE(unk_token="[UNK]")` | Model — handles out-of-vocabulary words |
| 2 | `Whitespace` pre-tokenizer | Splits input on whitespace before BPE merges |
| 3 | `BpeTrainer` | Learns merge rules to hit the target `vocab_size` |
| 4 | `train_from_iterator()` | Reads training data and builds the vocabulary |
| 5 | `model.save()` | Exports `vocab.json` and `merges.txt` for reuse |

Special tokens like `[UNK]`, `[CLS]`, `[SEP]`, `[PAD]`, and `[MASK]` are reserved slots in the vocabulary that models use for structural purposes (padding, sentence boundaries, masking for MLM, etc.).

> **Note:** Training on only 3 sentences is purely illustrative. A real tokenizer is trained on a large, representative corpus to produce a meaningful vocabulary.

In [15]:
from tokenizers import Tokenizer
from tokenizers.models import BPE # import byte pair encoding tokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Initialize a BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Set the pre-tokenizer to split on whitespace
tokenizer.pre_tokenizer = Whitespace()

# Example training data
training_data = [
    "Hello, how are you doing today?",
    "I am fine, thank you!",
    "What are your plans for the weekend?"
]

# Initialize a BPE trainer
trainer = BpeTrainer(vocab_size=1000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]) 

# Train the tokenizer on the training data
tokenizer.train_from_iterator(training_data, trainer)

# Save the tokenizer to a file
tokenizer.model.save(".", "custom_bpe_tokenizer")

['./custom_bpe_tokenizer-vocab.json', './custom_bpe_tokenizer-merges.txt']